In [1]:
"""
K-Nearest Neighbors, implemented from scratch.

Includes:
  - Distance functions: L1 (Manhattan), L2 (Euclidean), cosine, Minkowski
  - KNN            : classifier + regressor, brute-force search,
                     optional distance weighting
  - KDTree         : exact nearest-neighbor search via recursive
                     axis-aligned space partitioning. O(log n) average
                     query time in low dimensions; degrades toward
                     O(n) as dimensionality grows (see module notes).
  - standardize()  : zero-mean/unit-variance feature scaling
                     (required -- KNN is distance-based)

Smoke tests at the bottom exercise classification, regression, distance
metrics, and KD-tree-vs-brute-force agreement.
"""

import heapq
import math
import random



# Distance functions


def l2_distance(a, b):
    """Euclidean distance. Sensitive to feature scale -- standardize first."""
    return math.sqrt(sum((ai - bi) ** 2 for ai, bi in zip(a, b)))


def l1_distance(a, b):
    """Manhattan distance. More robust to outliers than L2."""
    return sum(abs(ai - bi) for ai, bi in zip(a, b))


def cosine_distance(a, b):
    """1 - cosine similarity. Ignores magnitude, only direction matters.
    Good default for text/embedding data."""
    dot_val = sum(ai * bi for ai, bi in zip(a, b))
    norm_a = math.sqrt(sum(ai ** 2 for ai in a))
    norm_b = math.sqrt(sum(bi ** 2 for bi in b))
    if norm_a == 0 or norm_b == 0:
        return 1.0
    return 1.0 - dot_val / (norm_a * norm_b)


def minkowski_distance(a, b, p=2):
    """Generalizes L1 (p=1) and L2 (p=2). p -> inf gives Chebyshev distance."""
    if p == float("inf"):
        return max(abs(ai - bi) for ai, bi in zip(a, b))
    return sum(abs(ai - bi) ** p for ai, bi in zip(a, b)) ** (1.0 / p)



# Feature scaling


def standardize(X):
    """Zero-mean, unit-variance scaling. Returns (X_scaled, means, stds).
    KNN needs this -- an unscaled large-range feature will dominate the
    distance calculation and drown out everything else."""
    n, d = len(X), len(X[0])
    means = [sum(row[j] for row in X) / n for j in range(d)]
    stds = []
    for j in range(d):
        var = sum((row[j] - means[j]) ** 2 for row in X) / n
        stds.append(math.sqrt(var) if var > 1e-12 else 1.0)
    X_scaled = [[(row[j] - means[j]) / stds[j] for j in range(d)] for row in X]
    return X_scaled, means, stds


def apply_scaling(X, means, stds):
    """Apply a previously-fit standardization (e.g. to test data)."""
    return [[(row[j] - means[j]) / stds[j] for j in range(len(row))] for row in X]



# KNN: classifier + regressor, brute-force search


class KNN:
    """
    K-nearest neighbors, brute-force (O(n*d) per query).

    task='classification' -> majority vote (or distance-weighted vote)
    task='regression'     -> mean of neighbor targets (or weighted mean)
    """

    def __init__(self, k=5, distance_fn=l2_distance, weighted=False,
                 task="classification"):
        self.k = k
        self.distance_fn = distance_fn
        self.weighted = weighted
        self.task = task
        self.X_train = None
        self.y_train = None

    def fit(self, X, y):
        # "Training" is just storing the data -- KNN is a lazy learner.
        self.X_train = X
        self.y_train = y
        return self

    def _neighbors(self, x):
        """Return list of (distance, index) for the k nearest training points."""
        distances = [(self.distance_fn(x, xi), i) for i, xi in enumerate(self.X_train)]
        distances.sort(key=lambda pair: pair[0])
        return distances[: self.k]

    def _predict_one(self, x):
        neighbors = self._neighbors(x)
        eps = 1e-10

        if self.task == "classification":
            if not self.weighted:
                votes = {}
                for _dist, idx in neighbors:
                    label = self.y_train[idx]
                    votes[label] = votes.get(label, 0) + 1
                return max(votes.items(), key=lambda item: item[1])[0]
            else:
                weighted_votes = {}
                for dist, idx in neighbors:
                    label = self.y_train[idx]
                    w = 1.0 / (dist + eps)
                    weighted_votes[label] = weighted_votes.get(label, 0.0) + w
                return max(weighted_votes.items(), key=lambda item: item[1])[0]

        elif self.task == "regression":
            if not self.weighted:
                values = [self.y_train[idx] for _dist, idx in neighbors]
                return sum(values) / len(values)
            else:
                num, denom = 0.0, 0.0
                for dist, idx in neighbors:
                    w = 1.0 / (dist + eps)
                    num += w * self.y_train[idx]
                    denom += w
                return num / denom if denom > 0 else 0.0

        raise ValueError(f"Unknown task: {self.task}")

    def predict(self, X):
        return [self._predict_one(x) for x in X]



# KD-tree: exact nearest-neighbor search via space partitioning


class _KDNode:
    __slots__ = ("point", "index", "left", "right", "axis")

    def __init__(self, point, index, axis, left=None, right=None):
        self.point = point
        self.index = index
        self.axis = axis
        self.left = left
        self.right = right


class KDTree:
    """
    Recursively partitions the data along feature axes (splitting on the
    median at each level), enabling nearest-neighbor queries faster than
    brute force -- in low dimensions.

    Average query time: O(log n) for d roughly under ~20.
    Degrades toward O(n) as dimensionality grows, because backtracking
    prunes fewer and fewer branches (curse of dimensionality). Past
    d ~ 20-50, brute force or a ball tree is often just as fast or faster.
    """

    def __init__(self, X, distance_fn=l2_distance):
        self.distance_fn = distance_fn
        self.n_dims = len(X[0]) if X else 0
        indices = list(range(len(X)))
        self.root = self._build(X, indices, depth=0)

    def _build(self, X, indices, depth):
        if not indices:
            return None

        axis = depth % self.n_dims
        indices = sorted(indices, key=lambda i: X[i][axis])
        median_pos = len(indices) // 2
        median_idx = indices[median_pos]

        node = _KDNode(point=X[median_idx], index=median_idx, axis=axis)
        node.left = self._build(X, indices[:median_pos], depth + 1)
        node.right = self._build(X, indices[median_pos + 1:], depth + 1)
        return node

    def query(self, point, k=1):
        """Return the k nearest (distance, index) pairs, sorted by distance."""
        # max-heap of size k, storing (-distance, index) so heap[0] is the
        # current worst of the k-best found so far -- lets us prune cheaply.
        heap = []

        def visit(node):
            if node is None:
                return

            dist = self.distance_fn(point, node.point)

            if len(heap) < k:
                heapq.heappush(heap, (-dist, node.index))
            elif dist < -heap[0][0]:
                heapq.heapreplace(heap, (-dist, node.index))

            axis = node.axis
            diff = point[axis] - node.point[axis]
            near_branch = node.left if diff < 0 else node.right
            far_branch = node.right if diff < 0 else node.left

            visit(near_branch)

            # Only descend into the far branch if it could still contain
            # a point closer than our current worst k-th best distance.
            worst_best = -heap[0][0] if len(heap) == k else float("inf")
            if abs(diff) < worst_best:
                visit(far_branch)

        visit(self.root)
        results = sorted(((-neg_dist, idx) for neg_dist, idx in heap), key=lambda p: p[0])
        return results


class KNNWithKDTree:
    """Same interface as KNN, but uses a KD-tree for the search step.
    Only sensible for L2 distance and low-to-moderate dimensionality."""

    def __init__(self, k=5, weighted=False, task="classification"):
        self.k = k
        self.weighted = weighted
        self.task = task
        self.tree = None
        self.y_train = None

    def fit(self, X, y):
        self.tree = KDTree(X, distance_fn=l2_distance)
        self.y_train = y
        return self

    def _predict_one(self, x):
        neighbors = self.tree.query(x, k=self.k)  # [(distance, index), ...]
        eps = 1e-10

        if self.task == "classification":
            weighted_votes = {}
            for dist, idx in neighbors:
                label = self.y_train[idx]
                w = 1.0 / (dist + eps) if self.weighted else 1.0
                weighted_votes[label] = weighted_votes.get(label, 0.0) + w
            return max(weighted_votes.items(), key=lambda item: item[1])[0]

        elif self.task == "regression":
            if not self.weighted:
                values = [self.y_train[idx] for _dist, idx in neighbors]
                return sum(values) / len(values)
            num, denom = 0.0, 0.0
            for dist, idx in neighbors:
                w = 1.0 / (dist + eps)
                num += w * self.y_train[idx]
                denom += w
            return num / denom if denom > 0 else 0.0

        raise ValueError(f"Unknown task: {self.task}")

    def predict(self, X):
        return [self._predict_one(x) for x in X]



# Helpers for smoke tests


def accuracy(y_true, y_pred):
    correct = sum(1 for a, b in zip(y_true, y_pred) if a == b)
    return correct / len(y_true)


def mse(y_true, y_pred):
    return sum((a - b) ** 2 for a, b in zip(y_true, y_pred)) / len(y_true)



# Smoke tests


if __name__ == "__main__":
    random.seed(42)

    # --- Classification: three well-separated 2D blobs ---
    X_cls, y_cls = [], []
    centers = [(-3, -3), (3, 3), (-3, 3)]
    for label, (cx, cy) in enumerate(centers):
        for _ in range(30):
            X_cls.append([random.gauss(cx, 0.6), random.gauss(cy, 0.6)])
            y_cls.append(label)

    knn = KNN(k=5, distance_fn=l2_distance, task="classification").fit(X_cls, y_cls)
    preds = knn.predict(X_cls)
    print("Brute-force KNN classification accuracy:", accuracy(y_cls, preds))

    knn_weighted = KNN(k=5, weighted=True, task="classification").fit(X_cls, y_cls)
    preds_w = knn_weighted.predict(X_cls)
    print("Distance-weighted KNN classification accuracy:", accuracy(y_cls, preds_w))

    # --- Regression: y = sin(x) + noise ---
    X_reg = [[x / 10.0] for x in range(-50, 50)]
    y_reg = [math.sin(x[0]) + random.gauss(0, 0.05) for x in X_reg]

    knn_reg = KNN(k=5, task="regression").fit(X_reg, y_reg)
    preds_reg = knn_reg.predict(X_reg)
    print("KNN regression MSE (unweighted, k=5):", round(mse(y_reg, preds_reg), 5))

    knn_reg_w = KNN(k=5, weighted=True, task="regression").fit(X_reg, y_reg)
    preds_reg_w = knn_reg_w.predict(X_reg)
    print("KNN regression MSE (weighted, k=5):", round(mse(y_reg, preds_reg_w), 5))

    # --- Distance metrics comparison ---
    a, b = [1.0, 2.0, 3.0], [4.0, 6.0, 3.0]
    print("\nDistance metric comparison for", a, "vs", b)
    print("  L1 (Manhattan):", l1_distance(a, b))
    print("  L2 (Euclidean):", round(l2_distance(a, b), 4))
    print("  Cosine:", round(cosine_distance(a, b), 4))
    print("  Minkowski (p=3):", round(minkowski_distance(a, b, p=3), 4))

    # --- KD-tree vs brute force: verify they agree ---
    brute = KNN(k=3, distance_fn=l2_distance, task="classification").fit(X_cls, y_cls)
    kdtree_knn = KNNWithKDTree(k=3, task="classification").fit(X_cls, y_cls)

    query_points = X_cls[:10]
    brute_preds = brute.predict(query_points)
    kdtree_preds = kdtree_knn.predict(query_points)
    agree = sum(1 for bp, kp in zip(brute_preds, kdtree_preds) if bp == kp)
    print(f"\nKD-tree vs brute-force agreement: {agree}/{len(query_points)} predictions match")

    # Verify the underlying neighbor sets actually match, not just the vote outcome
    tree = KDTree(X_cls, distance_fn=l2_distance)
    for x in query_points[:3]:
        kd_neighbors = sorted(idx for _dist, idx in tree.query(x, k=5))
        brute_neighbors = sorted(
            i for _d, i in sorted(
                [(l2_distance(x, xi), i) for i, xi in enumerate(X_cls)]
            )[:5]
        )
        assert kd_neighbors == brute_neighbors, "KD-tree neighbors do not match brute force!"
    print("KD-tree neighbor sets verified identical to brute force (k=5, first 3 queries)")


Brute-force KNN classification accuracy: 1.0
Distance-weighted KNN classification accuracy: 1.0
KNN regression MSE (unweighted, k=5): 0.00258
KNN regression MSE (weighted, k=5): 0.0

Distance metric comparison for [1.0, 2.0, 3.0] vs [4.0, 6.0, 3.0]
  L1 (Manhattan): 7.0
  L2 (Euclidean): 5.0
  Cosine: 0.1445
  Minkowski (p=3): 4.4979

KD-tree vs brute-force agreement: 10/10 predictions match
KD-tree neighbor sets verified identical to brute force (k=5, first 3 queries)
